# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

Select the project’s .venv kernel and choose **Run All**. The notebook indexes the first 100 movies and returns a sample match.


### One-time environment setup

The project environment is already configured. For a fresh Python 3.12 environment on this Intel Mac, run the following in the activated terminal before opening the notebook.

```sh
python -m pip install "torch==2.2.2" "numpy==1.26.4" "pandas<3" "scipy==1.15.3" "transformers==4.48.3" "sentence-transformers==3.4.1" ipywidgets ipykernel chromadb streamlit
```

In [3]:
from movie_search import PROJECT_DIR, collection_name, database, load_model, semantic_search

model = load_model()

In [4]:
import csv

dataset_path = PROJECT_DIR / "wiki_movie_plots_deduped.csv"
movie_limit = 100  # Set to None to process the full CSV.

tokenizer = model.tokenizer
chunk_size = model.max_seq_length - tokenizer.num_special_tokens_to_add(pair=False)
overlap = 32

texts = []
metadatas = []
ids = []

with dataset_path.open(encoding="utf-8-sig", newline="") as file:
    for row_id, movie in enumerate(csv.DictReader(file)):
        if movie_limit is not None and row_id >= movie_limit:
            break

        plot = movie["Plot"].strip()
        if not plot:
            continue

        # Slice the original plot at token boundaries to preserve its wording.
        offsets = tokenizer(
            plot,
            add_special_tokens=False,
            truncation=False,
            return_offsets_mapping=True,
            verbose=False,
        )["offset_mapping"]

        for start in range(0, len(offsets), chunk_size - overlap):
            end = min(start + chunk_size, len(offsets))
            texts.append(plot[offsets[start][0]:offsets[end - 1][1]])
            ids.append(f"movie-{row_id}-token-{start}")
            metadatas.append({
                "title": movie["Title"],
                "year": movie["Release Year"],
                "genre": movie["Genre"],
            })
            if end == len(offsets):
                break

print(f"Prepared {len(texts):,} plot chunks.")
if texts:
    print(metadatas[0]["title"], "→", texts[0][:200])

Prepared 135 plot chunks.
Kansas Saloon Smashers → A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man


In [5]:
# Generate one embedding per plot chunk.
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"{embeddings.shape[0]:,} chunks × {embeddings.shape[1]} dimensions")

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

135 chunks × 384 dimensions


In [6]:
# Save the index for the notebook and the Streamlit app.
client = database()
collection = client.get_or_create_collection(collection_name(movie_limit))
batch_size = min(1000, client.get_max_batch_size())
for start in range(0, len(ids), batch_size):
    batch = slice(start, start + batch_size)
    collection.upsert(
        ids=ids[batch],
        documents=texts[batch],
        embeddings=embeddings[batch],
        metadatas=metadatas[batch],
    )
print(f"Indexed {len(ids):,} plot chunks.")

Indexed 135 plot chunks.


In [7]:
query = "a train robbery"
movie, excerpt = semantic_search(query, model, collection)
print(f"{movie['title']} ({movie['year']})")
print(excerpt)

The Little Train Robbery (1905)
a trusted member proceed to a wild mountain spot in a bend of the railroad, where the road runs over a steep embankment. The spot is an ideal one for holding up a train. Cross ties are now placed on the railroad track and the gang hide in some bushes close by and wait for the train. The train soon approaches and is brought to a stop. The engineer leaves his engine and proceeds to remove the obstruction on the track. While he is bending over one of the gang sneaks up behind them and hits him on the head with an axe, and knocks him senseless down the embankment, while the gang surround the train and hold up the passengers. After securing all the "valuables," consisting principally of candy and dolls, the robbers uncouple the engine and one car and make their escape just in time to avoid a posse of police who appear on the scene. Further up the road they abandon the engine and car, take to the woods and soon reach their ponies.
In the meantime the police ha

### Open the Streamlit app

From the project folder, run `.venv/bin/python -m streamlit run app.py` in the terminal. The app searches the index saved above. To index the entire CSV, change `movie_limit` to `None` and rerun the notebook.

For the assignment, add your deployment requirements and publish the app on Hugging Face Spaces.